In [1]:
# packages
import pandas as pd
from mod02_build_bot_predictor import train_model

### Define a function to extract predictions from the model

In [2]:
def predict_bot(df, model=None):
    """
    Predict whether each account is a bot (1) or human (0).
    """
    if model is None:
        model = train_model()

    preds = model.predict(df)
    return pd.Series(preds, index=df.index)

### Define a function to evaluate model error

In [3]:
def confusion_matrix_and_metrics(y_true, y_pred):
    """
    Computes confusion matrix and common error rates for binary classification.

    Assumes labels:
      0 = negative class
      1 = positive class

    Returns:
      dict with:
        tn, fp, fn, tp
        misclassification_rate
        false_positive_rate
        false_negative_rate
    """
    tn = fp = fn = tp = 0

    for yt, yp in zip(y_true, y_pred):
        if yt == 0 and yp == 0:
            tn += 1
        elif yt == 0 and yp == 1:
            fp += 1
        elif yt == 1 and yp == 0:
            fn += 1
        elif yt == 1 and yp == 1:
            tp += 1
        else:
            raise ValueError("Labels must be 0 or 1")

    total = tn + fp + fn + tp

    misclassification_rate = (fp + fn) / total if total > 0 else 0.0
    false_positive_rate = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    false_negative_rate = fn / (fn + tp) if (fn + tp) > 0 else 0.0

    return {
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "misclassification_rate": misclassification_rate,
        "false_positive_rate": false_positive_rate,
        "false_negative_rate": false_negative_rate,
    }


### Load the data

In [4]:
TRAIN_PATH = "mod02_data/train.csv"
train = pd.read_csv(TRAIN_PATH)

TEST_PATH = "mod02_data/test.csv"
test = pd.read_csv(TEST_PATH)

### Format the data by independent vs. dependent variables

In [5]:
X_train = train.drop(columns=["is_bot"])
y_train = train['is_bot']

X_test = test.drop(columns=["is_bot"])
y_test = test['is_bot']

### Build the model on training data

In [6]:
model = train_model(X_train, y_train)

### Get the model predictions on training and test data

In [7]:
y_pred_train = predict_bot(X_train, model)
y_pred_test = predict_bot(X_test, model)

### Check results on the training set (data used to build the model)

In [8]:
confusion_matrix_and_metrics(y_train, y_pred_train)

{'tp': 117,
 'tn': 2596,
 'fp': 41,
 'fn': 246,
 'misclassification_rate': 0.09566666666666666,
 'false_positive_rate': 0.015547971179370497,
 'false_negative_rate': 0.6776859504132231}

### Check results on the test set (new data not yet seen by the model)

In [9]:
confusion_matrix_and_metrics(y_test, y_pred_test)

{'tp': 31,
 'tn': 861,
 'fp': 13,
 'fn': 95,
 'misclassification_rate': 0.108,
 'false_positive_rate': 0.014874141876430207,
 'false_negative_rate': 0.753968253968254}

# Discussion Questions

### Based on the misclassification rate of your model, discuss your confidence in the ability to predict a bot. 

The model has a missclassification rate of 10% for the test data and 9.5% misclassification rate for the training data. So that means that the misclassification rates are similar and that the classfication rate is around 90% which is pretty good. The values being similar means that the model isn't biased towards the training set, which is important. I think it's pretty good, but on a larger scale, like 1 million accounts, that means around 100K of them are being misclassified in some way, which isn't too good when you think about it at that scale.

### What are potential ramifications of false positives from the model?

The false positive rate for the training set is 1.55% and for the test set it's 1.48%. Flagging a real person as a bot punishes someone who didn't do anything wrong. For example, if a real human gets flagged, they could get locked out of their account, have their posts suppressed, or get shadowbanned with no clear way to appeal it, even though they did nothing to deserve it. I think certain groups could get hit harder by this too, like people whose posting patterns or response times just naturally look more "automated" (posting in bursts, replying really fast or really slow), even though there's nothing wrong with how they're using the platform. It's unfair to those people since they can't really fend for themselves against an automated decision.

### What are potential ramifications of false negatives from the model?

Potential ramifications of false negatives are that the undetected bot could cause harm to others over time through spam, manipulation, or fraud. This is really bad because it could falsely manipulate humans into clicking on something or interacting with it, thinking they're talking to another real person. Over time this can also mess with things like engagement numbers or ad metrics, since bots that slip through keep interacting with the platform undetected.